In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
X–Theta Framework: Version-0A (Static) vs Version-0B (Dynamic, Option B)
-----------------------------------------------------------------------

What this script provides (paper-ready):
  • Version-0A: static endpoint model with optional Bob twist omega.
  • Version-0B: path-dependent bounded memory h(t), updated ONLY on Bob transitions (Option B).
  • Reproducible comparison: prints E_ij and CHSH S for 0A, 0B-CW, 0B-CCW.
  • Significance: replicate ΔS across runs + sign-flip permutation p-value.
  • Scan: δ sweep -> CSV + plot (S_CW, S_CCW, ΔS) with Tsirelson + classical bounds.
  • Saves:
      - x_theta_0A_0B_comparison.csv
      - x_theta_v0B_dS_replicates.csv
      - x_theta_v0B_scan.csv

Conventions:
  • CHSH form used: S = E00 + E01 + E10 - E11
  • With the standard angles below and singlet-like sign, S ≈ -2√2 at omega=0.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------
# Utilities
# ----------------------------
def wrap_pi(x: float) -> float:
    return (x + np.pi) % (2.0 * np.pi) - np.pi


def tsirelson() -> float:
    return 2.0 * np.sqrt(2.0)


# ----------------------------
# CHSH angles (Tsirelson max at omega=0 for our sign convention)
# ----------------------------
A_SETTINGS = [0.0, np.pi / 2.0]  # a0, a1
B_SETTINGS = [np.pi / 4.0, -np.pi / 4.0]  # b0, b1

# CHSH plaquette corner ordering (a_idx, b_idx)
# 0:(a0,b0) -> 1:(a0,b1) -> 2:(a1,b1) -> 3:(a1,b0) -> back
CORNERS = [(0, 0), (0, 1), (1, 1), (1, 0)]


# ----------------------------
# Physics: target correlator
# ----------------------------
def E_target(a: float, b: float, beta_eff: float, v: float = 1.0) -> float:
    """
    X–Theta target correlation function:
      E = -v * cos( a - (b + beta_eff) )
    """
    return float(-v * np.cos(a - (b + beta_eff)))


# ----------------------------
# No-signaling ±1 outcomes via correlated Gaussian thresholding
# ----------------------------
def sample_outcome_product(E_t: float, rng: np.random.Generator) -> int:
    """
    Produces an AB product with expectation E_t and flat marginals.
    """
    E_t = float(np.clip(E_t, -1.0, 1.0))
    rho = np.sin((np.pi / 2.0) * (-E_t))

    z1 = rng.standard_normal()
    z2 = rng.standard_normal()
    zB = rho * z1 + np.sqrt(max(0.0, 1.0 - rho * rho)) * z2

    A_out = 1 if z1 >= 0 else -1
    B_out = -(1 if zB >= 0 else -1)  # singlet-like anticorrelation
    return A_out * B_out


# ----------------------------
# Version-0A: Static endpoint model
# ----------------------------
def simulate_v0A_static(
    n_samples: int = 500_000, v: float = 1.0, omega: float = 0.0, seed: int = 123
):
    """
    Static model:
      beta_base(b0)=0
      beta_base(b1)=omega
    Sampling is interleaved across (a,b) bins for symmetry.
    """
    rng = np.random.default_rng(seed)
    beta_base = {0: 0.0, 1: wrap_pi(omega)}

    sum_ab = np.zeros((2, 2), dtype=np.float64)
    cnt_ab = np.zeros((2, 2), dtype=np.int64)

    for _ in range(n_samples):
        ai = int(rng.integers(0, 2))
        bj = int(rng.integers(0, 2))

        a = A_SETTINGS[ai]
        b = B_SETTINGS[bj]
        beta_eff = beta_base[bj]

        E_t = E_target(a, b, beta_eff, v=v)
        sum_ab[ai, bj] += sample_outcome_product(E_t, rng)
        cnt_ab[ai, bj] += 1

    E = np.divide(sum_ab, np.maximum(cnt_ab, 1))
    S = float(E[0, 0] + E[0, 1] + E[1, 0] - E[1, 1])

    return {
        "S": S,
        "E": E,
        "cnt": cnt_ab,
        "omega": float(beta_base[1]),
        "n_samples": int(n_samples),
        "v": float(v),
        "seed": int(seed),
    }


# ----------------------------
# Version-0B: Dynamic loop (Option B: Bob-only bounded memory)
# ----------------------------
def simulate_v0B_dynamic(
    mode: str = "cw",
    steps: int = 2500,
    dwell: int = 200,
    delta: float = 0.004,
    leak_edge: float = 0.05,
    v: float = 1.0,
    omega: float = 0.0,
    seed: int = 123,
):
    """
    Dynamic model (Option B):
      beta_eff(a,b,t) = beta_base(b) + h(t)
      beta_base(b0)=0, beta_base(b1)=omega

    Memory update occurs ONLY when Bob changes b (bj != bj_next):
      h <- (1 - leak_edge) * h + sgn * delta
    (We also apply decay on Alice edges to keep the same relaxation timescale.)
    """
    rng = np.random.default_rng(seed)
    mode = mode.lower()
    if mode not in ("cw", "ccw"):
        raise ValueError("mode must be 'cw' or 'ccw'")

    sgn = +1.0 if mode == "cw" else -1.0
    beta_base = {0: 0.0, 1: wrap_pi(omega)}

    sum_ab = np.zeros((2, 2), dtype=np.float64)
    cnt_ab = np.zeros((2, 2), dtype=np.int64)

    h = 0.0
    c = 0

    for _ in range(steps):
        ai, bj = CORNERS[c]

        # Measure at current corner
        a = A_SETTINGS[ai]
        b = B_SETTINGS[bj]
        beta_eff = wrap_pi(beta_base[bj] + h)

        for _ in range(dwell):
            E_t = E_target(a, b, beta_eff, v=v)
            sum_ab[ai, bj] += sample_outcome_product(E_t, rng)
            cnt_ab[ai, bj] += 1

        # Move to next corner (defines the operational path)
        if mode == "cw":
            c_next = (c + 1) % 4
        else:
            c_next = (c - 1) % 4

        _, bj_next = CORNERS[c_next]

        # Option-B: update h only on Bob transitions; always decay
        if bj != bj_next:
            h = wrap_pi((1.0 - leak_edge) * h + sgn * delta)
        else:
            h = wrap_pi((1.0 - leak_edge) * h)

        c = c_next

    E = np.divide(sum_ab, np.maximum(cnt_ab, 1))
    S = float(E[0, 0] + E[0, 1] + E[1, 0] - E[1, 1])

    return {
        "S": S,
        "E": E,
        "cnt": cnt_ab,
        "h_final": float(h),
        "omega": float(beta_base[1]),
        "steps": int(steps),
        "dwell": int(dwell),
        "delta": float(delta),
        "leak_edge": float(leak_edge),
        "v": float(v),
        "seed": int(seed),
        "mode": mode,
    }


# ----------------------------
# Significance: replicate ΔS + sign-flip permutation test
# ----------------------------
def replicate_asymmetry(
    R: int = 200,
    steps: int = 2000,
    dwell: int = 150,
    delta: float = 0.004,
    leak_edge: float = 0.05,
    v: float = 1.0,
    omega: float = 0.0,
    seed0: int = 77_000,
    perm_rounds: int = 5000,
):
    """
    Runs R paired CW/CCW simulations with different seeds.
    Returns mean ΔS, sd(ΔS), z-score for the mean, and sign-flip permutation p-value.
    """
    dS = np.zeros(R, dtype=np.float64)

    for r in range(R):
        cw = simulate_v0B_dynamic(
            mode="cw",
            steps=steps,
            dwell=dwell,
            delta=delta,
            leak_edge=leak_edge,
            v=v,
            omega=omega,
            seed=seed0 + 2 * r,
        )
        ccw = simulate_v0B_dynamic(
            mode="ccw",
            steps=steps,
            dwell=dwell,
            delta=delta,
            leak_edge=leak_edge,
            v=v,
            omega=omega,
            seed=seed0 + 2 * r + 1,
        )
        dS[r] = cw["S"] - ccw["S"]

    mu = float(dS.mean())
    sd = float(dS.std(ddof=1))
    z = float(mu / (sd / np.sqrt(R))) if sd > 0 else np.inf

    # Sign-flip permutation (null: ΔS equally likely +/-)
    obs = abs(mu)
    rng = np.random.default_rng(seed0 + 999)
    flips = rng.choice([-1.0, +1.0], size=(perm_rounds, R))
    perm_means = np.abs((flips * dS[None, :]).mean(axis=1))
    p = float((perm_means >= obs).mean())

    return {
        "R": int(R),
        "mean_dS": mu,
        "sd_dS": sd,
        "z_of_mean": z,
        "p_value_signflip": p,
        "dS_samples": dS,
        "steps": int(steps),
        "dwell": int(dwell),
        "delta": float(delta),
        "leak_edge": float(leak_edge),
        "v": float(v),
        "omega": float(omega),
        "perm_rounds": int(perm_rounds),
    }


# ----------------------------
# Scan δ -> CSV
# ----------------------------
def scan_params_and_save(
    deltas=np.linspace(0.0, 0.012, 25),
    leak_edge: float = 0.05,
    steps: int = 2000,
    dwell: int = 150,
    v: float = 1.0,
    omega: float = 0.0,
    out_csv: str = "x_theta_v0B_scan.csv",
    seed0: int = 88_000,
):
    rows = []
    for k, delta in enumerate(deltas):
        cw = simulate_v0B_dynamic(
            mode="cw",
            steps=steps,
            dwell=dwell,
            delta=float(delta),
            leak_edge=leak_edge,
            v=v,
            omega=omega,
            seed=seed0 + 2 * k,
        )
        ccw = simulate_v0B_dynamic(
            mode="ccw",
            steps=steps,
            dwell=dwell,
            delta=float(delta),
            leak_edge=leak_edge,
            v=v,
            omega=omega,
            seed=seed0 + 2 * k + 1,
        )
        rows.append(
            {
                "delta": float(delta),
                "leak_edge": float(leak_edge),
                "S_cw": float(cw["S"]),
                "S_ccw": float(ccw["S"]),
                "dS": float(cw["S"] - ccw["S"]),
                "h_final_cw": float(cw["h_final"]),
                "h_final_ccw": float(ccw["h_final"]),
                "N0B": int(cw["cnt"].sum()),
                "steps": int(steps),
                "dwell": int(dwell),
                "omega": float(omega),
                "v": float(v),
            }
        )
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")
    return df


# ----------------------------
# Plotting
# ----------------------------
def plot_bar(res0a, rescw, resccw):
    labels = ["Static (0A)", "CW (0B)", "CCW (0B)"]
    vals = [res0a["S"], rescw["S"], resccw["S"]]

    plt.figure(figsize=(7, 5))
    plt.bar(labels, vals, alpha=0.75)

    t = tsirelson()
    plt.axhline(+t, linestyle="--", label="Tsirelson ±2√2")
    plt.axhline(-t, linestyle="--")
    plt.axhline(+2.0, linestyle="--", label="Classical ±2")
    plt.axhline(-2.0, linestyle="--")

    plt.ylabel("CHSH S-value")
    plt.title("X-Theta Model Comparison: Version-0A vs Version-0B (Option B)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_scan(df):
    plt.figure(figsize=(10, 6))
    plt.plot(df["delta"], df["S_cw"], label="S_CW")
    plt.plot(df["delta"], df["S_ccw"], label="S_CCW")
    plt.plot(df["delta"], df["dS"], label="ΔS = S_CW - S_CCW")

    t = tsirelson()
    plt.axhline(+t, linestyle="--", label="Tsirelson ±2√2")
    plt.axhline(-t, linestyle="--")
    plt.axhline(+2.0, linestyle="--", label="Classical ±2")
    plt.axhline(-2.0, linestyle="--")

    plt.xlabel("edge kick δ (radians)")
    plt.ylabel("CHSH S and asymmetry")
    plt.title("Version-0B (Option B): CW/CCW asymmetry vs δ")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ----------------------------
# Main: run everything
# ----------------------------
if __name__ == "__main__":
    print("=== X-Theta Framework: Version-0A & 0B (Option B) + significance ===")

    # --- Parameters (edit these freely) ---
    V = 1.0
    OMEGA = 0.0

    # 0A
    N0A = 500_000
    SEED_0A = 123

    # 0B
    STEPS = 2500
    DWELL = 200
    DELTA = 0.004
    LEAK_EDGE = 0.05
    SEED_CW = 123
    SEED_CCW = 456

    # --- Run 0A ---
    res0a = simulate_v0A_static(n_samples=N0A, v=V, omega=OMEGA, seed=SEED_0A)
    print("\n[Version-0A Static]")
    print(
        f"omega={res0a['omega']:.6f}  S={res0a['S']:+.6f}  target≈{-tsirelson():+.6f}"
    )
    E0 = res0a["E"]
    print(
        f"E00={E0[0,0]:+.4f} E01={E0[0,1]:+.4f} E10={E0[1,0]:+.4f} E11={E0[1,1]:+.4f}"
    )

    # --- Run 0B CW/CCW ---
    rescw = simulate_v0B_dynamic(
        mode="cw",
        steps=STEPS,
        dwell=DWELL,
        delta=DELTA,
        leak_edge=LEAK_EDGE,
        v=V,
        omega=OMEGA,
        seed=SEED_CW,
    )
    resccw = simulate_v0B_dynamic(
        mode="ccw",
        steps=STEPS,
        dwell=DWELL,
        delta=DELTA,
        leak_edge=LEAK_EDGE,
        v=V,
        omega=OMEGA,
        seed=SEED_CCW,
    )

    print("\n[Version-0B Dynamic - Option B (Bob-only memory)]")
    print(
        f"params: steps={STEPS} dwell={DWELL} delta={DELTA} leak_edge={LEAK_EDGE} omega={OMEGA}"
    )
    print(f"CW  S={rescw['S']:+.6f}  h_final={rescw['h_final']:+.6f}")
    print(f"CCW S={resccw['S']:+.6f}  h_final={resccw['h_final']:+.6f}")
    print(f"Asymmetry ΔS = (CW-CCW) = {(rescw['S'] - resccw['S']):+.6f}")

    # --- Save one-row summary CSV ---
    comp = pd.DataFrame(
        [
            {
                "S_static": res0a["S"],
                "S_cw": rescw["S"],
                "S_ccw": resccw["S"],
                "dS_cw_minus_ccw": rescw["S"] - resccw["S"],
                "delta": DELTA,
                "leak_edge": LEAK_EDGE,
                "omega": OMEGA,
                "v": V,
                "N0A": N0A,
                "N0B": int(rescw["cnt"].sum()),
                "h_final_cw": rescw["h_final"],
                "h_final_ccw": resccw["h_final"],
            }
        ]
    )
    comp.to_csv("x_theta_0A_0B_comparison.csv", index=False)
    print("\nSaved: x_theta_0A_0B_comparison.csv")

    # --- Significance of ΔS (replicates + sign-flip p) ---
    stats = replicate_asymmetry(
        R=200,
        steps=2000,
        dwell=150,
        delta=DELTA,
        leak_edge=LEAK_EDGE,
        v=V,
        omega=OMEGA,
        seed0=77_000,
        perm_rounds=5000,
    )
    print("\n[ΔS significance over replicates]")
    print(
        f"R={stats['R']}  mean(ΔS)={stats['mean_dS']:+.6f}  sd(ΔS)={stats['sd_dS']:.6f}"
    )
    print(
        f"z(mean)={stats['z_of_mean']:+.3f}  p(sign-flip)={stats['p_value_signflip']:.6g}"
    )

    pd.DataFrame({"dS": stats["dS_samples"]}).to_csv(
        "x_theta_v0B_dS_replicates.csv", index=False
    )
    print("Saved: x_theta_v0B_dS_replicates.csv")

    # --- δ scan -> CSV + plot ---
    df = scan_params_and_save(
        deltas=np.linspace(0.0, 0.012, 25),
        leak_edge=LEAK_EDGE,
        steps=2000,
        dwell=150,
        v=V,
        omega=OMEGA,
        out_csv="x_theta_v0B_scan.csv",
        seed0=88_000,
    )
    plot_bar(res0a, rescw, resccw)
    plot_scan(df)

=== X-Theta Framework: Version-0A & 0B (Option B) + significance ===

[Version-0A Static]
omega=0.000000  S=-2.831838  target≈-2.828427
E00=-0.7084 E01=-0.7078 E10=-0.7068 E11=+0.7088

[Version-0B Dynamic - Option B (Bob-only memory)]
params: steps=2500 dwell=200 delta=0.004 leak_edge=0.05 omega=0.0
CW  S=-2.827120  h_final=+0.038974
CCW S=-2.830272  h_final=-0.041026
Asymmetry ΔS = (CW-CCW) = +0.003152

Saved: x_theta_0A_0B_comparison.csv


KeyboardInterrupt: 